# 🎸 Regresión Simbólica — Ley de la cuerda vibrante

## Objetivo

Redescubrir desde los datos experimentales la ecuación:

$$\boxed{f = \frac{1}{2L}\sqrt{\frac{T}{\mu}}}$$

donde:
- $f$ = frecuencia fundamental [Hz]
- $L$ = longitud vibrante de la cuerda [m]
- $T$ = tensión en la cuerda [N]
- $\mu$ = densidad lineal (masa por unidad de longitud) [kg/m]

## Datos disponibles

| Experimento | Variables que cambian | Variables fijas |
|---|---|---|
| Trastes (0–6) por cuerda | $L$, $f$ | $T$, $\mu$ (por cuerda) |
| Clavijero (−180° a +180°) por cuerda | $T$, $f$ | $L$ = 65 cm, $\mu$ (por cuerda) |

Juntos cubren variación independiente en las tres variables de entrada.

---
## Celda 1 – Instalación y carga de librerías

> PySR requiere Julia en el sistema. La primera vez puede tardar varios minutos en compilar.

In [ ]:
# Instala PySR si no está disponible
import subprocess, sys
try:
    import pysr
    print(f'✅ PySR ya instalado — versión {pysr.__version__}')
except ImportError:
    print('Instalando PySR...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pysr', '-q'])
    import pysr
    print(f'✅ PySR instalado — versión {pysr.__version__}')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pysr import PySRRegressor

plt.rcParams.update({
    'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.4,
    'axes.titlesize': 12, 'axes.labelsize': 11, 'legend.fontsize': 9
})
print('✅ Todas las librerías cargadas.')

---
## Celda 2 – Lectura y limpieza del dataset

In [ ]:
ARCHIVO = 'dataset.csv'  # ← ajusta si es necesario

df_raw = pd.read_csv(
    ARCHIVO,
    sep=';',
    decimal=',',          # decimales con coma
    skipinitialspace=True,
    skip_blank_lines=True,
    encoding='latin-1'   # soporta tildes
)

# Limpiar nombres de columna
df_raw.columns = ['cuerda', 'traste_ang', 'frecuencia', 'tension', 'densidad', 'longitud']

# Limpiar espacios/tabs dentro de valores de texto que quedaron como string
for col in ['frecuencia', 'tension', 'densidad', 'longitud']:
    if df_raw[col].dtype == object:
        df_raw[col] = df_raw[col].astype(str).str.replace(r'\s+', '', regex=True).str.replace(',', '.')
        df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

df_raw['traste_ang'] = pd.to_numeric(df_raw['traste_ang'], errors='coerce')
df_raw = df_raw.dropna().reset_index(drop=True)

print(f'✅ Dataset cargado: {len(df_raw)} filas, {df_raw["cuerda"].nunique()} cuerdas')
print(df_raw.to_string(index=True))

---
## Celda 3 – Conversión de unidades al SI

La ecuación teórica requiere unidades del Sistema Internacional:
- $L$: metros (el CSV está en **cm** → dividir entre 100)
- $\mu$: kg/m (el CSV ya está en kg/m)
- $T$: N (el CSV ya está en N)
- $f$: Hz (el CSV ya está en Hz)

In [ ]:
df = df_raw.copy()

# Convertir longitud de cm → m
df['L_m']    = df['longitud'] / 100.0
df['T_N']    = df['tension']
df['mu_kgm'] = df['densidad']
df['f_Hz']   = df['frecuencia']

# Etiquetar el tipo de experimento
es_traste = df['traste_ang'].isin(range(0, 7))
df['experimento'] = np.where(es_traste, 'traste', 'clavijero')

# Eliminar duplicados (traste 0 y clavijero 0° son la misma medición)
df = df.drop_duplicates(subset=['cuerda', 'L_m', 'T_N', 'mu_kgm']).reset_index(drop=True)

print(f'✅ Conversión completada. Filas finales: {len(df)}')
print(f'   Trastes:    {(df["experimento"]=="traste").sum()} filas')
print(f'   Clavijero:  {(df["experimento"]=="clavijero").sum()} filas')
print()
print(df[['cuerda','experimento','f_Hz','T_N','mu_kgm','L_m']].round(5).to_string(index=True))

---
## Celda 4 – Verificación del modelo teórico sobre los datos

Antes de la regresión simbólica, verificamos que la ecuación teórica $f = \frac{1}{2L}\sqrt{\frac{T}{\mu}}$ ya describe bien los datos.

In [ ]:
# Predicción teórica
df['f_teorica'] = (1 / (2 * df['L_m'])) * np.sqrt(df['T_N'] / df['mu_kgm'])

# Error relativo
df['error_pct'] = np.abs(df['f_Hz'] - df['f_teorica']) / df['f_Hz'] * 100

print('Comparación f_medida vs f_teórica:')
print(df[['cuerda','experimento','f_Hz','f_teorica','error_pct']]
      .rename(columns={'f_Hz':'f_medida [Hz]','f_teorica':'f_teórica [Hz]','error_pct':'error [%]'})
      .round(3).to_string(index=True))
print(f'\nError relativo medio: {df["error_pct"].mean():.2f}%')
print(f'Error relativo máximo: {df["error_pct"].max():.2f}%')

# Gráfica paridad
fig, ax = plt.subplots(figsize=(6, 6))
colores = plt.cm.tab10(np.linspace(0, 0.6, 6))
for c in range(1, 7):
    sub = df[df['cuerda'] == c]
    ax.scatter(sub['f_Hz'], sub['f_teorica'], label=f'Cuerda {c}',
               color=colores[c-1], s=60, edgecolors='k', lw=0.4)

lims = [df['f_Hz'].min() * 0.95, df['f_Hz'].max() * 1.05]
ax.plot(lims, lims, 'k--', lw=1.5, label='Identidad (perfecto)')
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel('f medida [Hz]'); ax.set_ylabel('f teórica [Hz]')
ax.set_title('Paridad: modelo teórico vs datos experimentales')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('paridad_teorica.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Guardada 'paridad_teorica.png'")

---
## Celda 5 – Preparación del conjunto de entrenamiento para PySR

In [ ]:
# Variables de entrada (features) y variable objetivo
X = df[['T_N', 'mu_kgm', 'L_m']].to_numpy()
y = df['f_Hz'].to_numpy()

print(f'✅ Conjunto de entrenamiento listo.')
print(f'   X shape: {X.shape}  →  columnas: [T_N, mu_kgm, L_m]')
print(f'   y shape: {y.shape}  →  f_Hz')
print()
print(f'   Rangos de las variables:')
for nombre, col in zip(['T_N', 'mu_kgm', 'L_m'], X.T):
    print(f'   {nombre:8s}: [{col.min():.5f}, {col.max():.5f}]')
print(f'   f_Hz    : [{y.min():.3f}, {y.max():.3f}]')

---
## Celda 6 – Regresión simbólica con PySR

> **Tiempo estimado:** 2–5 minutos dependiendo del hardware.  
> PySR busca expresiones matemáticas usando algoritmos evolutivos.  
> Los operadores habilitados (`sqrt`, `*`, `/`) son los que aparecen en la ecuación teórica.

In [ ]:
model = PySRRegressor(
    # ── Operadores permitidos ──────────────────────────────────────
    binary_operators   = ["+", "-", "*", "/"],
    unary_operators    = ["sqrt"],          # raíz cuadrada es clave

    # ── Criterio de selección ──────────────────────────────────────
    # "accuracy"  → prioriza el mejor ajuste
    # "best"      → equilibrio entre ajuste y simplicidad (recomendado)
    model_selection    = "best",

    # ── Complejidad ────────────────────────────────────────────────
    niterations        = 100,     # iteraciones evolutivas
    populations        = 30,      # poblaciones paralelas
    maxsize            = 20,      # máx. nodos en el árbol de expresión
    parsimony          = 0.001,   # penalización por complejidad

    # ── Nombres de columnas para que la ecuación sea legible ───────
    # (se asignan en el orden de X)
    extra_sympy_mappings = {},

    verbosity          = 1,
    random_state       = 42,
    temp_equation_file = True,
)

# Asignar nombres a las columnas
import pandas as pd
X_df = pd.DataFrame(X, columns=['T', 'mu', 'L'])

print('🔍 Iniciando búsqueda simbólica...')
model.fit(X_df, y)
print('\n✅ Búsqueda completada.')

---
## Celda 7 – Resultados: tabla de ecuaciones encontradas

In [ ]:
# Tabla completa: complejidad vs ecuación vs R²
print('=' * 70)
print('  ECUACIONES ENCONTRADAS POR PYSR (de menor a mayor complejidad)')
print('=' * 70)
print(model.equations_[['complexity', 'equation', 'loss', 'score']].to_string(index=True))
print()
print('─' * 70)
print(f'  Ecuación seleccionada como MEJOR:')
print(f'  {model.sympy()}')
print('─' * 70)

---
## Celda 8 – Gráfica: Curva de Pareto (complejidad vs error)

In [ ]:
eqs = model.equations_.copy()
eqs['rmse'] = np.sqrt(eqs['loss'])

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(eqs['complexity'], eqs['rmse'], 'o-', color='steelblue',
        markersize=8, lw=2)

# Resaltar la ecuación seleccionada
best_idx = model.equations_['score'].idxmax()
ax.scatter(eqs.loc[best_idx, 'complexity'], eqs.loc[best_idx, 'rmse'],
           color='crimson', s=150, zorder=5, label='Ecuación seleccionada')

# Anotar ecuaciones cortas
for i, row in eqs.iterrows():
    eq_str = str(row['equation'])
    if len(eq_str) < 45:
        ax.annotate(eq_str, (row['complexity'], row['rmse']),
                    textcoords='offset points', xytext=(6, 4),
                    fontsize=7, color='gray')

ax.set_xlabel('Complejidad de la expresión (nodos)')
ax.set_ylabel('RMSE [Hz]')
ax.set_title('Curva de Pareto: complejidad vs precisión\n(esquina inferior izquierda = ideal)')
ax.set_yscale('log')
ax.legend()
plt.tight_layout()
plt.savefig('pareto_pysr.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Guardada 'pareto_pysr.png'")

---
## Celda 9 – Gráfica: f predicha vs f medida (paridad)

In [ ]:
f_pred = model.predict(X_df)
f_teo  = (1 / (2 * df['L_m'])) * np.sqrt(df['T_N'] / df['mu_kgm'])

# R² manual
ss_res = np.sum((y - f_pred)**2)
ss_tot = np.sum((y - y.mean())**2)
R2 = 1 - ss_res / ss_tot

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Comparación de modelos — f [Hz]', fontsize=13)

colores = plt.cm.tab10(np.linspace(0, 0.6, 6))

for ax_idx, (f_modelo, titulo, color_line) in enumerate([
    (f_pred, f'PySR — ecuación simbólica\n$R^2 = {R2:.5f}$', 'crimson'),
    (f_teo,  f'Modelo teórico $\\frac{{1}}{{2L}}\\sqrt{{T/\\mu}}$\n$R^2 = {1 - np.sum((y-f_teo.values)**2)/ss_tot:.5f}$', 'steelblue'),
]):
    ax = axes[ax_idx]
    for c in range(1, 7):
        mask = df['cuerda'].values == c
        ax.scatter(y[mask], np.array(f_modelo)[mask],
                   label=f'Cuerda {c}', color=colores[c-1], s=55,
                   edgecolors='k', lw=0.4)
    lims = [y.min()*0.93, y.max()*1.05]
    ax.plot(lims, lims, 'k--', lw=1.5)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel('f medida [Hz]'); ax.set_ylabel('f predicha [Hz]')
    ax.set_title(titulo)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('paridad_comparacion.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Guardada paridad_comparacion.png')
print(f'   R² PySR:     {R2:.6f}')
print(f'   R² teórico:  {1 - np.sum((y-f_teo.values)**2)/ss_tot:.6f}')

---
## Celda 10 – Panel: f vs cada variable (T, μ, L) separada por cuerda

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Frecuencia vs variables físicas — datos experimentales', fontsize=13)

variables = [
    ('T_N',    'Tensión T [N]'),
    ('mu_kgm', 'Densidad lineal μ [kg/m]'),
    ('L_m',    'Longitud vibrante L [m]'),
]
colores = plt.cm.tab10(np.linspace(0, 0.6, 6))

for ax, (col, xlabel) in zip(axes, variables):
    for c in range(1, 7):
        sub = df[df['cuerda'] == c].sort_values(col)
        ax.scatter(sub[col], sub['f_Hz'], color=colores[c-1],
                   s=55, edgecolors='k', lw=0.4, label=f'C{c}', zorder=4)
        ax.plot(sub[col], sub['f_Hz'], color=colores[c-1],
                lw=1, alpha=0.5, zorder=3)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('f [Hz]')
    ax.set_title(f'f vs {xlabel.split(" ")[0]}')

axes[0].legend(fontsize=8, title='Cuerda')
plt.tight_layout()
plt.savefig('panel_f_vs_variables.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Guardada 'panel_f_vs_variables.png'")

---
## Celda 11 – Ecuación final y comparación numérica

In [ ]:
print('=' * 65)
print('  RESUMEN FINAL')
print('=' * 65)
print()
print('  Ecuación teórica esperada:')
print('  f = (1 / (2·L)) · √(T / μ)')
print()
print('  Ecuación encontrada por PySR:')
print(f'  f = {model.sympy()}')
print()

# Tabla de errores por cuerda
df['f_pysr'] = f_pred
resumen = df.groupby('cuerda').apply(lambda g: pd.Series({
    'n_datos': len(g),
    'f_media_Hz': g['f_Hz'].mean(),
    'RMSE_teo_Hz': np.sqrt(np.mean((g['f_Hz'] - g['f_teorica'])**2)),
    'RMSE_pysr_Hz': np.sqrt(np.mean((g['f_Hz'] - g['f_pysr'])**2)),
    'err_teo_%': (np.abs(g['f_Hz'] - g['f_teorica']) / g['f_Hz'] * 100).mean(),
    'err_pysr_%': (np.abs(g['f_Hz'] - g['f_pysr']) / g['f_Hz'] * 100).mean(),
})).reset_index()

print('  Error por cuerda:')
print(resumen.round(4).to_string(index=False))

print()
print(f'  RMSE global — Teórico:  {np.sqrt(np.mean((y - f_teo.values)**2)):.4f} Hz')
print(f'  RMSE global — PySR:     {np.sqrt(np.mean((y - f_pred)**2)):.4f} Hz')

---
## Celda 12 – Exportar resultados

In [ ]:
# Dataset con predicciones incluidas
df_out = df[['cuerda','experimento','f_Hz','T_N','mu_kgm','L_m',
             'f_teorica','f_pysr','error_pct']].copy()
df_out['err_pysr_pct'] = np.abs(df['f_Hz'] - df['f_pysr']) / df['f_Hz'] * 100
df_out.to_csv('resultados_regresion_simbolica.csv', index=False, float_format='%.6f')

# Tabla de ecuaciones de Pareto
model.equations_[['complexity','equation','loss','score']].to_csv(
    'ecuaciones_pareto.csv', index=True)

print('✅ Archivos exportados:')
print('   - resultados_regresion_simbolica.csv')
print('   - ecuaciones_pareto.csv')

---
## Notas sobre la interpretación

### ¿Qué esperar de PySR?

Si los datos son consistentes con la física, PySR debería encontrar algo equivalente a:

$$f = \frac{\sqrt{T/\mu}}{2L} = \frac{1}{2} \cdot L^{-1} \cdot T^{0.5} \cdot \mu^{-0.5}$$

Puede aparecer en formas algebráicamente equivalentes como `sqrt(T/mu) / (2*L)` o `0.5 * sqrt(T) / (L * sqrt(mu))`.

### Curva de Pareto
La gráfica de Pareto muestra el **balance complejidad–precisión**. La ecuación óptima está en el "codo" de esa curva — donde agregar más complejidad ya no mejora significativamente el error.

### Fuentes de desviación
- Las cuerdas reales tienen **rigidez de flexión** (stiffness), que desplaza ligeramente las frecuencias respecto al modelo ideal.
- Los valores de $\mu$ tomados de internet pueden diferir de los valores reales de tus cuerdas.
- El ángulo de la clavija es una medida indirecta de $T$: cualquier error en la calibración de $T$ se propaga.